# 05 · Ensamble

Combina los boostings (LightGBM, XGBoost, CatBoost) con TabPFN (`kaggle_tabpfn/07_tabpfn_kaggle.ipynb`, corre en Kaggle con GPU).

1. Los tres boostings se promedian en partes iguales (están correlacionados al 0.995; ponderarlos no cambia nada).
2. Ese promedio se combina con TabPFN. El único peso a elegir (cuánto pesa TabPFN) se busca en una grilla chica sobre las predicciones *out-of-fold*, nunca con test.

In [1]:
import os, time, numpy as np, pandas as pd, warnings
import features as F
warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
y = F.load_y(); folds = F.cv_folds(y)

BOOST = ["lightgbm", "xgboost", "catboost"]
oof = {n: np.load(f"{F.CACHE}/oof_{n}.npy") for n in BOOST}
tst = {n: np.load(f"{F.CACHE}/test_{n}.npy") for n in BOOST}
z = np.load("kaggle_tabpfn/output_v3/tabpfn.npz")   # corrida con las features del vendedor (4 estimadores, semilla 42)
oof["tabpfn"], tst["tabpfn"] = z["oof"], z["test"]
acc = lambda p: ((p > .5) == y).mean()

In [2]:
boost_oof, boost_tst = np.mean([oof[n] for n in BOOST], axis=0), np.mean([tst[n] for n in BOOST], axis=0)
tabla = {"baseline": acc(np.load(f"{F.CACHE}/oof_baseline.npy"))}
tabla.update({n: acc(oof[n]) for n in oof})
tabla["promedio boostings"] = acc(boost_oof)
print(pd.Series(tabla).round(4).to_string())
print("\nCorrelación entre modelos:"); print(pd.DataFrame(oof).corr().round(4))

baseline              0.8234
lightgbm              0.9474
xgboost               0.9470
catboost              0.9475
tabpfn                0.9478
promedio boostings    0.9475

Correlación entre modelos:
          lightgbm  xgboost  catboost  tabpfn
lightgbm    1.0000   0.9986    0.9966  0.9938
xgboost     0.9986   1.0000    0.9955  0.9924
catboost    0.9966   0.9955    1.0000  0.9939
tabpfn      0.9938   0.9924    0.9939  1.0000


In [3]:
pesos = pd.Series({w: acc(w * oof["tabpfn"] + (1 - w) * boost_oof) for w in np.arange(0, 1.01, .1).round(1)})
print("accuracy CV según el peso de TabPFN:"); print(pesos.round(4).to_string())

accuracy CV según el peso de TabPFN:
0.0    0.9475
0.1    0.9478
0.2    0.9480
0.3    0.9481
0.4    0.9485
0.5    0.9486
0.6    0.9485
0.7    0.9484
0.8    0.9484
0.9    0.9482
1.0    0.9478


La curva es plana entre 0.4 y 0.7, así que se usa el punto medio (0.5): mitad TabPFN, mitad boostings. Elegir el máximo exacto de la grilla sería sobreajustar a estas predicciones.

In [4]:
W = 0.5
print(f"peso TabPFN = {W} | accuracy CV: {acc(W * oof['tabpfn'] + (1 - W) * boost_oof):.4f}")
proba = W * tst["tabpfn"] + (1 - W) * boost_tst
np.save(f"{F.CACHE}/test_ensamble.npy", proba)
F.save_submission(tst["tabpfn"], "tabpfn")
F.save_submission(proba, "ensamble")["condition"].value_counts(normalize=True)

peso TabPFN = 0.5 | accuracy CV: 0.9486


condition
new     0.5263
used    0.4737
Name: proportion, dtype: float64